# ICA on CI EEG Data
This notebook loads the dataset and applies Independent Component Analysis (ICA) using MNE to try to separate CI artifacts.

In [ ]:
import json
import matplotlib.pyplot as plt
import mne
import numpy as np
import zarr
from mne.preprocessing import ICA

mne.set_log_level('WARNING')
ZARR_PATH = 'D:/ci_denoise/eeg_raw.zarr'

In [ ]:
def zarr_to_raw(group):
    data = np.array(group['data'])
    sfreq = float(group.attrs['sfreq'])
    ch_names = list(group.attrs['ch_names'])

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)

    onsets    = np.array(group['annotations_onset'])
    durations = np.array(group['annotations_duration'])
    descs     = json.loads(group.attrs['annotations_description'])
    n = min(len(onsets), len(descs))
    raw.set_annotations(mne.Annotations(
        onset=onsets[:n], duration=durations[:n], description=descs[:n],
    ))

    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore', match_case=False, verbose=False)
    return raw

def first_block_window(raw, on_code='10', off_code='11'):
    onsets = raw.annotations.onset
    descs = list(map(str, raw.annotations.description))
    t10 = next((onsets[i] for i in range(len(descs)) if descs[i] == on_code), None)
    t11 = next((onsets[i] for i in range(len(descs))
                if descs[i] == off_code and onsets[i] > (t10 if t10 is not None else -1)), None)
    if t10 is None or t11 is None:
        return None
    return float(t10), float(t11)

def crop_to_first_block(raw):
    w = first_block_window(raw)
    if w is None:
        return raw
    t0, t1 = w
    return raw.copy().crop(tmin=t0, tmax=t1, verbose=False)

store = zarr.open(ZARR_PATH, mode='r')
CI_NAME = 'noise0_perm1'
raw_ci_full = zarr_to_raw(store['ci'][CI_NAME])
raw_ci = crop_to_first_block(raw_ci_full)

In [ ]:
# Initialize and fit ICA
# (Data is already 2Hz high-passed, which is good for ICA)
ica = ICA(n_components=15, max_iter='auto', random_state=97)
ica.fit(raw_ci)

In [ ]:
# Plot the independent components' topographies
ica.plot_components()

In [ ]:
# Plot the sources to identify artifact components manually over time
ica.plot_sources(raw_ci)

In [ ]:
# After identifying the artifact components visually, you can exclude them:
# ica.exclude = [0, 1]  # Replace with actual artifact component indices
# raw_clean = raw_ci.copy()
# ica.apply(raw_clean)
# raw_clean.plot()